# Laboratory Day 9b: Semantic Segmentation (U-Net): Modeling, Training

In this exercise, we will implement and train a U-Net using Tensorlow. In the last exercise, we have prepared tfrecords, which should be used in this exercise.

For deep learning task, there is usually no unique solution. Feel free when you write your own implementation.

To solve the task here, all functions in Tensorflow Lib are allowed be used/called.


## 1. Modeling and Loss Function
**Exercise 1.1 (10 points)**
1. Build the U-Net structure from scratch and print out model.summary().
2. Implement the cross-entropy loss function. 

In [ ]:
#  code sample. Feel free to change it
from keras.layers import Conv2D, MaxPooling2D, Input
from keras.models import Model

def modeling(img_h, img_w, img_c, num_class):
    x_input = Input(shape=(img_h, img_w, img_c))


    y_out = layers(x_input)

    # modeling
    model = Model(inputs=x_input, outputs=y_out)
    model.summary()

    return model

In [ ]:
def cross_entropy_loss(image_mask, y_pred):
    # todo: call functions in Tensorflow
    pass

## 2. Training
**Exercise 2.1 (20 points)**:
1. Write a training program to train the model for in total $N$ steps. Choose the proper learning rate such that the loss function can be effectively reduced.

2. For each $S$ training steps (e.g. $S$=1000),

    - Print out (a) the average loss of the training dataset for the last $S$ steps.
    
    - Optional: Compute the confusion matrix (based on the classification results of each pixel)
       

3. After each $S$ training steps, evaluate the average loss of the validation data set.

    - Print out (a) the average loss of the entire validation dataset.
    
    - Optional: Compute the confusion matrix. Compute precision, recall as metrics.
    
   
4. Please print out all debugging info and metrics evaluation results into a *.txt file. It is recommended to use logging package for recording longer training process (optional, see below).

5. Optional task: One can also visualize the training process in tensorboard.

    - Tensorboard can be started by running this command:  tensorboard --logdir="path_to_dir" --bind_all
    
    - Using tensorboard is recommended, but not mandatory.
    

6. Implement save_model() and load_model() to save and load the model.


U-Net is not a trivial system, and training it is complicated. Any mistakes in the data pipeline, in the modeling and in the selection of training parameters may lead to undesired training results. To debug the training process, my suggestions:

1. Visualize the images and the image masks, which are directly feed into network.

2. Visualize the output of the network. If the network learns, the predicted mask should converge to the GT mask.

3. Obverse the loss function of training set and validation set, which should get minimized by the optimizer.

4. Be careful of overfitting. Use data augmentation, batch normalization, regularization, dropout layers, early stop to fight against overfitting.

4. Check the implementation of the cross-entropy loss function.

5. Adjust the learning rate. Small learning rates may result in a more stable training process.

Hope you could get a working U-Net.

In [ ]:
# functions, which could be used
import logging
def init_logging(log_path=None, mode='w', level=logging.INFO):
    if not log_path:
        log_path = os.path.join(os.getcwd(), 'filename.log')
    if level is None:
        level = logging.INFO
    format = '%(asctime)s - %(name)s - %(levelname)s : %(message)s'
    handlers = [logging.FileHandler(log_path, mode=mode), logging.StreamHandler()]
    logging.basicConfig(level=level, format=format, handlers=handlers)
    logging.info(f'log_path: {log_path}')


In [ ]:
# Solution template
import logging
import tensorflow as tf
import numpy as np

work_dir = r'' # #todo

# logging
logging_path = os.path.join(work_dir, 'logging.log')
init_logging(logging_path, level=logging.DEBUG)
logger = logging.getLogger(__name__)


class UnetModel(tf.keras.Model):
    def __init__(self, work_dir):
        super().__init__()
        self.input_w = 572
        self.input_h = 572
        self.input_c = 3
        self.num_class = 2

        self._setup_layers()
        self._setup_metrics()

        self.work_dir = work_dir

    @property
    def metrics(self):
        return [self._loss_metric]

    def _setup_metrics(self):
        self._loss_metric = tf.keras.metrics.Mean(name='loss_mean')
        # self._conf_matrix =  #optional


    def _metrics_reset(self):
        for m in self.metrics:
            m.reset_states()

    def _update_metrics(self, loss):
        self._loss_metric.update_state(loss)


    def _setup_layers(self):
        self.cnn = modeling(...)

    def train(self, data_train, steps, delta_steps=100):
        for step, (image, y_label) in enumerate(data_train):

            with tf.GradientTape() as tape:
                y_pred = ... # Forward pass
                loss = self.loss_fun(y_pred, y_label) # compute loss

            trainable_vars = self.trainable_variables
            gradients = tape.gradient(loss, trainable_vars)
            self.optimizer.apply_gradients(zip(gradients, trainable_vars))

            # update metrics
            self._update_metrics(loss)

            if step % delta_steps == 0:
                self._metrics_reset()
                # todo: evaluate val.tfrecord
                self.data_visualize() #optional: for debug
                self.savemodel()


            if step >= steps:
                break

        logging.info(f'done: total_step {step}, finish')


    def loss_fun(self, y_pred, y_label):
        # re-use the code
        raise NotImplemented

    def data_visualize(self, data):
        raise NotImplemented

    def save_model():
        raise NotImplemented

    def load_model():
        raise NotImplemented



model = UnetModel(...)
model.train(...)

## 3. Prediction
**Exercise 3.1 (5 points)**:
1. Load the U-Net model by the best Checkpoint or SavedModel, which are obtained after training.
2. Do prediction on 2-3 new images, which have not seen by the model. Visualize the image mask and prediction.

In [ ]:
# Solution template
checkpoint_file = r''

model = UnetModel()
model.load_model(checkpoint_file)

prediction = model.prediction(image_new) # or model(image_new)
